Before you begin, open this experiment on Trovi:

-   Use this link: [Large-scale model training on Chameleon](https://trovi.chameleoncloud.org/dashboard/artifacts/bd06bd6d-d94f-4297-ad5d-c9b7e1f02575) on Trovi
-   Then, click “Launch on Chameleon”. This will start a new Jupyter server for you, with the experiment materials already in it.

Inside the `llm-chi` directory, open the `single` subdirectory. You will see several notebooks - look for the one titled `2_create_server.ipynb`. Open this notebook and continue there.

## Connect to Chameleon project and site

At the beginning of the lease time, we will bring up our GPU server. We will use the `python-chi` Python API to Chameleon to provision our server.

We will execute the cells in this notebook inside the Chameleon Jupyter environment.

In [ ]:
# run in Chameleon Jupyter environment
import chi, os, time
from chi import server, context, lease, network

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

Change the string in the following cell to reflect the name of *your* lease (**with your own net ID**), then run it to get your lease:

In [ ]:
# run in Chameleon Jupyter environment
LEASE_NAME = "bestshot-train-proj19"
l = lease.get_lease(LEASE_NAME)
l.show()

The status should show as “ACTIVE” now that we are past the lease start time.

The rest of this notebook sets up the instance and the experiment environment, which cumulatively takes a while - bringing up the instance can take some time, and building the container image can take some time.

But, you can be mostly hands-off in this stage. You can save time by clicking on this cell, then selecting Run \> Run Selected Cell and All Below from the Jupyter menu.

As the notebook executes, monitor its progress to make sure it does not get stuck on any execution error, and also to see what it is doing!

We will use the lease to bring up a server with the `CC-Ubuntu24.04-CUDA` disk image.

The default boot disk for instances at KVM@TACC is a little small for large model training, so we will first create a larger boot volume (200 GiB) from that image, then boot the server from that volume.

## Bring up the server

Creates a 100 GiB boot volume from `CC-Ubuntu24.04-CUDA` and launches the server.
If the server already exists, this cell skips creation.

EfficientNet-B3 is much smaller than LLMs so 100 GiB is plenty (the lab used 200 GiB for BLIP-2).

In [ ]:
# run in Chameleon Jupyter environment
username = os.getenv('USER') # all exp resources will have this suffix
server_name = f"bestshot-gpu-proj19"

try:
    s = server.get_server(server_name)
    print(f"Server {server_name} already exists. Skipping create.")
except Exception:
    os_conn = chi.clients.connection()
    cinder_client = chi.clients.cinder()

    images = list(os_conn.image.images(name="CC-Ubuntu24.04-CUDA"))
    image_id = images[0].id

     # 100 GiB is enough for EfficientNet-B3 + KonIQ-10k (~5 GB images)
    boot_vol = cinder_client.volumes.create(
        name=f"boot-vol-bestshot-proj19",
        size=100,
        imageRef=image_id,
    )

    while True:
        boot_vol = cinder_client.volumes.get(boot_vol.id)
        if boot_vol.status == "available":
            break
        if boot_vol.status in ["error", "error_restoring", "error_extending"]:
            raise RuntimeError(f"Boot volume provisioning failed with status {boot_vol.status}")
        time.sleep(10)

    bdm = [{
        "boot_index": 0,
        "uuid": boot_vol.id,
        "source_type": "volume",
        "destination_type": "volume",
        "delete_on_termination": True,
    }]

    server_from_vol = os_conn.compute.create_server(
        name=server_name,
        flavor_id=server.get_flavor_id(l.get_reserved_flavors()[0].name),
        block_device_mapping_v2=bdm,
        networks=[{"uuid": os_conn.network.find_network("sharednet1").id}],
    )

    os_conn.compute.wait_for_server(server_from_vol, wait=600)
    s = server.get_server(server_name)
    print(f"Server {server_name} is ready.")

## Security Groups

We need security groups to allow SSH and Jupyter access. Open SSH (22) and MLflow UI (8000) ports.

In [ ]:
# run in Chameleon Jupyter environment
security_groups = [
  {'name': "allow-ssh", 'port': 22, 'description': "Enable SSH traffic on TCP port 22"},
  #{'name': "allow-8888", 'port': 8888, 'description': "Enable TCP port 8888 (used by Jupyter)"},
  {'name': 'allow-8000', 'port': 8000, 'description': 'Enable MLflow UI on port 8000'},
  {'name': 'allow-9000', 'port': 9000, 'description': 'Enable MinIO on port 9000'},
]

In [5]:
# run in Chameleon Jupyter environment
for sg in security_groups:
  secgroup = network.SecurityGroup({
      'name': sg['name'],
      'description': sg['description'],
  })
  secgroup.add_rule(direction='ingress', protocol='tcp', port=sg['port'])
  secgroup.submit(idempotent=True)
  s.add_security_group(sg['name'])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")

## Associate floating IP and verify connection

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In the output below, make a note of the floating IP that has been assigned to your instance.

In [ ]:
# Note the floating IP — you'll use it to access MLflow in your browser
s.refresh()
s.show(type="widget")

## Install Docker + NVIDIA container toolkit

In [ ]:
s.execute("curl -sSL https://get.docker.com | sudo sh")

In [ ]:
s.execute("groupadd -f docker && sudo usermod -aG docker $USER")

In [ ]:
s.execute("""
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey \
    | gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg && \
curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list \
    | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' \
    | tee /etc/apt/sources.list.d/nvidia-container-toolkit.list && \
apt-get -qq update && \
apt-get -qq install -y nvidia-container-toolkit
""")

In [ ]:
# Fix cgroup driver — required for NVIDIA + Docker to work together
s.execute("""
nvidia-ctk runtime configure --runtime=docker && \
systemctl restart docker
""")

In [ ]:
# Verify GPU is visible inside Docker — should show your H100
s.execute("docker run --rm --gpus all nvidia/cuda:12.0-base-ubuntu22.04 nvidia-smi")

## Clone BestShot repo

In [ ]:
REPO_URL = "https://github.com/anokhimehta/bestshot"

s.execute(f"git clone {REPO_URL} ~/bestshot")

## Download KonIQ-10k Dataset

Downloading directly to the Chameleon instance.
Using 512x384 images (~767 MB) — sufficient for EfficientNet-B3 training.

Source: University of Konstanz VQA group (https://database.mmsp-kn.de/koniq-10k-database.html)

In [ ]:
s.execute("mkdir -p ~/data/koniq10k")

In [ ]:
# Download images (512x384) — ~767 MB, takes a few minutes
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_512x384.zip
""")

In [ ]:
# Download scores CSV — ~304 KB
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Unzip both
s.execute("""
cd ~/data/koniq10k && \
unzip -q koniq10k_512x384.zip && \
unzip -q koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Verify — should show 10,073 images and the scores CSV
s.execute("ls ~/data/koniq10k/ && echo '---' && ls ~/data/koniq10k/512x384/ | wc -l")

In [ ]:
# run in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("docker run hello-world")

## Start MLflow

In [ ]:
s.execute("""
cd ~/bestshot && \
docker compose -f infrastructure/docker/docker-compose-mlflow.yaml up -d
""")

In [ ]:
# Verify MLflow is running
s.execute("docker ps")

In [ ]:
# Print the MLflow URL
s.refresh()
floating_ip = [addr['addr'] for addr in s.addresses.get('sharednet1', [])
               if addr['OS-EXT-IPS:type'] == 'floating'][0]
print(f"MLflow UI: http://{floating_ip}:8000")
print(f"MinIO UI:  http://{floating_ip}:9000")
print(f"\nSSH:       ssh cc@{floating_ip}")

## Build training container
Builds the Docker image from `training/Dockerfile` and `training/requirements.txt` in your repo.
This takes a few minutes the first time — subsequent builds are faster due to layer caching.

In [ ]:
# run in Chameleon Jupyter environment
s.execute("docker build -t bestshot-train:latest ~/bestshot/training/")

## Run training job

In [ ]:
s.refresh()
floating_ip = [addr['addr'] for addr in s.addresses.get('sharednet1', [])
               if addr['OS-EXT-IPS:type'] == 'floating'][0]

s.execute(f"""
sudo docker run --rm --gpus all \\
    -v ~/data/koniq10k:/data/koniq10k \\
    -v ~/bestshot:/workspace \\
    -w /workspace \\
    -e MLFLOW_TRACKING_URI=http://{floating_ip}:8000 \\
    bestshot-train:latest \\
    python training/train.py --config training/config/baseline.yaml
""")